# Notebook 07: Mask2Former & Module Capstone

**Module:** 09 Instance Segmentation  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand query-based universal segmentation
2. Know Mask2Former vs Panoptic FPN
3. Choose strategy for GeoSpatial projects
4. Complete Module 09

## Mask2Former (Cheng et al., 2022)

**Universal segmentation** — one model for semantic, instance, and panoptic via **mask classification**:

1. Pixel decoder produces high-res features
2. Transformer decoder with N learned queries
3. Each query → class logits + mask logits
4. Hungarian matching assigns queries to GT segments

**Advantage over Panoptic FPN:** Single unified architecture, no merge heuristics.

*(Also covered in Module 07 Notebook 13 — here we focus on instance/panoptic use.)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Architecture Comparison

| Model | Type | Speed | mAP/mPQ | Best For |
|-------|------|-------|---------|----------|
| UNet++ dual-head | Semantic + boundary | Fast | High IoU (your project) | Adjacent ponds |
| Mask R-CNN | Instance | Medium | Mask AP | Instance IDs needed |
| YOLACT | Instance | Fast | Good | Real-time video |
| SOLOv2 | Instance | Fast | Good | No proposals |
| Panoptic FPN | Panoptic | Medium | PQ | Cityscapes-style |
| Mask2Former | Universal | Medium | SOTA PQ | Research / unified model |

## water-bodies-detection: Architectural Decision

**Problem:** Adjacent aquaculture ponds merge in semantic segmentation.

**Option A — Your dual-head (Module 07):**
- Train aqua + bund boundary heads
- Post-process: watershed on boundaries → separate polygons
- Labels: semantic masks (simpler annotation)

**Option B — Mask R-CNN (Module 09):**
- Annotate each pond with instance ID
- Direct instance masks at inference
- Heavier labels, heavier model

**Option C — Hybrid:**
- UNet++ for boundaries + connected components for instance IDs
- No Mask R-CNN needed for many aquaculture use cases

**Recommendation for your project:** Dual-head is the right engineering choice. Know Mask R-CNN for when clients need instance tracking or panoptic dashboards.

## Module 09 Assignment

Implement connected-components instance labeling from binary aqua mask + boundary mask (simulate your post-process). Compare to Mask R-CNN on synthetic adjacent circles.

See `exercises/README.md`.

In [ ]:
# Connected components instance labeling (concept)
from scipy import ndimage
aqua = np.zeros((64, 128), dtype=np.uint8)
aqua[:, 15:30] = 1; aqua[:, 35:50] = 1  # two adjacent ponds
labeled, n = ndimage.label(aqua)
print(f'Instances found: {n}')
plt.imshow(labeled, cmap='nipy_spectral'); plt.title('Connected components'); plt.show()

## Module 09 Complete

**Next:** Module 10 Transformers — attention, ViT, BERT, GPT.

---

## Summary

Mask2Former unifies segmentation tasks; your dual-head boundary design is a pragmatic alternative to instance seg.